In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Set display options to show all rows and columns with full width
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)  # Show full content of each cell
# Filtering Q nur wenn accepted answer received in Phase 2,

df = pd.read_parquet('../03_processed_datasets/chunk_01.parquet')

print(df.columns.tolist())

# note: agg_df["meaned_numHelped"] = agg_df.groupby("user_id")["numHelped"].transform(lambda x: x - x.mean())
# 1. Model: numHelped
# 2. Model: Question FE: question_id demeanen
# 3. Model: User FE: user_id demeanen

# "meaned_numHelped ~ C(phase) + C(has_accepted_answer) + C(phase):C(has_accepted_answer)"
# 1x ohne controls, 1x mit allen

In [15]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Set display options to show all rows and columns with full width
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)  # Show full content of each cell

# Load the data
df = pd.read_parquet('../03_processed_datasets/chunk_01.parquet')

# (df.columns.tolist())

df = df.convert_dtypes()  # Convert columns to best possible dtypes
for col in df.select_dtypes(include=["Int32", "Int64"]).columns:  # Find nullable integer columns
    df[col] = df[col].astype("int64")  # Convert to standard int64
for col in df.select_dtypes(include=["Float32"]).columns:  # Handle nullable floats if needed
    df[col] = df[col].astype("float64")  # Convert to standard float64

df['has_helped'] = (df['numHelped'] > 0).astype(int)
df['ln_numHelped'] = np.log(df['numHelped'] + 1)

# Fixed effects for numHelped
df['user_fe_numHelped'] = df.groupby("user_id")["numHelped"].transform(lambda x: x - x.mean())
df['question_fe_numHelped'] = df.groupby("event_id")["numHelped"].transform(lambda x: x - x.mean())

# Fixed effects for has_helped
df['user_fe_has_helped'] = df.groupby("user_id")["has_helped"].transform(lambda x: x - x.mean())
df['question_fe_has_helped'] = df.groupby("event_id")["has_helped"].transform(lambda x: x - x.mean())

# Fixed effects for ln_numHelped
df['user_fe_ln_numHelped'] = df.groupby("user_id")["ln_numHelped"].transform(lambda x: x - x.mean())
df['question_fe_ln_numHelped'] = df.groupby("event_id")["ln_numHelped"].transform(lambda x: x - x.mean())

KeyboardInterrupt: 

In [13]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Define the formula for the three models
formula1 = "numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
formula2 = "user_fe_numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
formula3 = "question_fe_numHelped ~ C(phase) + C(phase):C(has_answer)"

# Fit the models with clustered standard errors
models = []
model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

for formula in formulas:
   # Fit the model
   model = smf.ols(formula=formula, data=df).fit()

   # Apply clustered standard errors by user_id
   model = model.get_robustcov_results(
       cov_type='cluster',
       groups=df['user_id']
   )

   models.append(model)

# Create and customize the Stargazer table
stargazer = Stargazer(models)
stargazer.title("Effect of Receiving Answers on Providing Help")
stargazer.custom_columns(model_names, [1, 1, 1])
stargazer.significant_digits(3)
stargazer.show_degrees_of_freedom(False)
stargazer.show_model_numbers(False)


# Print the table in a format suitable for LaTeX
# print(stargazer.render_latex())

# Print HTML version for easier viewing in notebooks
html_output = stargazer.render_html()
display(HTML(html_output))

C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '


In [11]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Define the formula for the three models
formula1 = "numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
formula2 = "user_fe_numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
formula3 = "question_fe_numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"

# Model names
model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

# Run and display each model individually
for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"\n\n==== {name} ====")

    # Fit the model
    model = smf.ols(formula=formula, data=df).fit()

    # Apply clustered standard errors by user_id
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=df['user_id']
    )

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Receiving Answers on Providing Help - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))



==== Base Model ====




==== User FE ====




==== Question FE ====


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '
